# FurEver Cinematic Generator — Lumi × Noctra

This Colab notebook builds a ~90-second FurEver cinematic from your Lumi and Noctra reference images using the distilled LTX-Video 2B image-to-video model. It saves every generated shot to Google Drive, so interrupted free Colab sessions can resume instead of starting over.

**Target:** 704×384, 24 fps, 17 generated shots plus a clean title card. Software and model usage are free; free Colab GPU availability is not guaranteed.


In [ ]:
#@title 1. GPU check + install
import subprocess, sys
try:
    print(subprocess.check_output(["nvidia-smi","--query-gpu=name,memory.total,driver_version","--format=csv,noheader"],text=True))
except Exception:
    raise RuntimeError("Attach a GPU from Runtime > Change runtime type > GPU, then reconnect.")
%pip -q install -U "diffusers==0.40.0" "transformers>=4.57.0" "accelerate>=1.10.0" "bitsandbytes>=0.48.0" "imageio[ffmpeg]>=2.37.0" "sentencepiece>=0.2.0" "safetensors>=0.6.0"
print("Video stack installed.")

In [ ]:
#@title 2. Load the FurEver generator module
import urllib.request, importlib.util
URL="https://raw.githubusercontent.com/imdhanxx-dk/FurEver/main/tools/furever_cinematic_colab.py"
LOCAL="/content/furever_cinematic_colab.py"
urllib.request.urlretrieve(URL,LOCAL)
spec=importlib.util.spec_from_file_location("furever_cinematic",LOCAL)
fc=importlib.util.module_from_spec(spec); spec.loader.exec_module(fc)
print(f"Loaded {len(fc.STORY)}-shot FurEver storyboard.")

In [ ]:
#@title 3. Mount Drive + add your two reference images
ROOT,REFS,SHOTS,FINAL=fc.mount_project()
fc.upload_reference(REFS,"noctra")
fc.upload_reference(REFS,"lumi")
REF_MAP=fc.prepare_refs(REFS)
print("References ready:",REF_MAP)
print("All outputs will be stored under:",ROOT)

In [ ]:
#@title 4. Load LTX-Video 2B distilled
PIPE=fc.load_pipeline()

In [ ]:
#@title 5. Generation settings
MODE="full" #@param ["preview","full"]
STEPS=8 #@param {type:"slider",min:6,max:12,step:1}
GUIDANCE=1.0 #@param {type:"slider",min:1.0,max:2.0,step:0.1}
print("Mode:",MODE,"| Steps:",STEPS,"| Guidance:",GUIDANCE)
if MODE=="preview": print("Preview generates only the first two short test shots.")
else: print("Full mode generates all 17 shots and resumes around any existing completed files.")

In [ ]:
#@title 6. Generate / resume shots
fc.generate(PIPE,REF_MAP,SHOTS,mode=MODE,steps=STEPS,guidance=GUIDANCE)

In [ ]:
#@title 7. Assemble and play the final ~90-second cinematic
from IPython.display import Video,display
if MODE!="full":
    print("Switch MODE to 'full', generate all shots, then run this cell.")
else:
    final_video=fc.assemble(ROOT,SHOTS,FINAL)
    display(Video(str(final_video),embed=False,width=900))

## Notes
- The notebook uses image-to-video anchors for character identity and short shot-reverse-shot editing to reduce drift.
- The prompts explicitly prioritize coherent feline anatomy and natural paw/limb motion.
- The final title is rendered with FFmpeg rather than AI-generated text, so the typography stays clean.
- The master is intentionally silent. Narration and music can be added as a separate pass without regenerating the expensive video shots.
